In [ ]:
# MUST be the first executed cell: %pip install restarts the Python kernel
# and wipes any variables defined above. Install deps BEFORE setting anything.
%pip install -q faker==30.* tqdm

# 00_seed_historical_data

Populates the **source systems** (Azure SQL + ADLS Gen2 `raw` container) with a fiscal quarter of Contoso Tech retail activity.

- **Azure SQL**: INSERT into all retail.* tables via JDBC, using the workspace user's AAD token.
- **ADLS Gen2 `raw`**: write dated daily files in BOTH CSV and Parquet for supplier feeds and marketing exports.

This notebook is invoked by `deploy.ps1` on first deploy. Re-running is idempotent (TRUNCATEs SQL tables before inserting; overwrites ADLS files).

## Parameters

Injected by `deploy.ps1` via the Fabric `RunNotebook` job parameters. Defaults below let the notebook be run interactively in the Fabric UI for ad-hoc re-seeding.

In [ ]:
sql_server_fqdn   = ""   # e.g. contoso-retail-sql-abc123.database.windows.net
sql_database_name = "contoso_retail"
storage_account   = ""   # e.g. contosortabc12345
raw_container     = "raw"

# Volume (fiscal quarter)
n_customers = 5_000
n_products  = 1_500
n_orders    = 50_000

# Seed window (last 90 days ending today)
import datetime
seed_end   = datetime.date.today()
seed_start = seed_end - datetime.timedelta(days=90)

random_seed = 42

## Setup

In [ ]:
import datetime
import random
import uuid
from typing import Iterator

from faker import Faker
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import *  # noqa

fake = Faker("en_US")
Faker.seed(random_seed)
random.seed(random_seed)

print(f"Seeding window: {seed_start} to {seed_end} ({(seed_end-seed_start).days} days)")
print(f"Volume: {n_customers:,} customers, {n_products:,} products, {n_orders:,} orders")
print(f"SQL: {sql_server_fqdn}/{sql_database_name}")
print(f"ADLS: abfss://{raw_container}@{storage_account}.dfs.core.windows.net/")

## Reference data (categories, brands, suppliers, warehouses, stores)

Hand-curated lists — these are the structural backbone every other table depends on.

In [ ]:
# Categories: (id, parent, name, path, sort)
CATEGORIES = [
    (1,  None, "Electronics",   "Electronics", 0),
    (2,  1,    "Smartphones",   "Electronics > Smartphones", 1),
    (3,  1,    "Laptops",       "Electronics > Laptops", 2),
    (4,  1,    "Tablets",       "Electronics > Tablets", 3),
    (5,  1,    "Headphones",    "Electronics > Headphones", 4),
    (6,  1,    "Smart Watches", "Electronics > Smart Watches", 5),
    (7,  1,    "Cameras",       "Electronics > Cameras", 6),
    (8,  1,    "Smart Home",    "Electronics > Smart Home", 7),
    (9,  1,    "Gaming",        "Electronics > Gaming", 8),
    (10, 9,    "Consoles",      "Electronics > Gaming > Consoles", 9),
    (11, 9,    "Accessories",   "Electronics > Gaming > Accessories", 10),
    (12, 1,    "Accessories",   "Electronics > Accessories", 11),
]

BRANDS = [
    (1, "Apex", "USA", True), (2, "Nimbus", "USA", True),
    (3, "Voltcraft", "Germany", False), (4, "Pixelworks", "USA", False),
    (5, "Skyline", "Korea", True), (6, "Foundry", "USA", False),
    (7, "Tundra", "Sweden", False), (8, "Kestrel", "USA", False),
    (9, "Lumen", "Japan", True), (10, "Echelon", "USA", False),
]

SUPPLIERS = [
    (i+1, f"{fake.company()} Supply", fake.company_email(), random.choice(["USA","China","Vietnam","Mexico","Korea"]), random.randint(3,30))
    for i in range(15)
]

WAREHOUSES = [
    (1, "Reno DC",       "Reno",        "NV", "USA", 250000),
    (2, "Atlanta DC",    "Atlanta",     "GA", "USA", 200000),
    (3, "Dallas DC",     "Dallas",      "TX", "USA", 180000),
    (4, "NJ DC",         "Edison",      "NJ", "USA", 150000),
    (5, "Chicago DC",    "Chicago",     "IL", "USA", 150000),
]

STORES = [
    (1, "Contoso Tech – Downtown SF",   "flagship", "401 Market St",  "San Francisco", "CA", "94105", "USA", "2019-06-15", 12000, "Alex Chen"),
    (2, "Contoso Tech – Manhattan",     "flagship", "485 5th Ave",    "New York",      "NY", "10017", "USA", "2020-03-20", 14000, "Mia Patel"),
    (3, "Contoso Tech – Lincoln Park",  "standard", "2110 N Clark",   "Chicago",       "IL", "60614", "USA", "2021-09-01",  8000, "Jordan Reyes"),
    (4, "Contoso Tech – Galleria",      "standard", "5085 Westheimer","Houston",       "TX", "77056", "USA", "2022-02-18",  7500, "Sasha Kim"),
    (5, "Contoso Tech – Outlet Reno",   "outlet",   "5500 Meadowood", "Reno",          "NV", "89502", "USA", "2023-11-04",  5000, "Devin Brooks"),
]

PROMOTIONS = [
    (1, "SUMMER25",    "Summer Sale 25% Off",       "percent",       25, 50),
    (2, "SAVE50",      "$50 Off Orders Over $500",  "fixed",         50, 500),
    (3, "FREESHIP",    "Free Shipping Sitewide",    "free_shipping", 0, 0),
    (4, "FALL15",      "Fall 15% Off",              "percent",       15, 0),
    (5, "HOLIDAY20",   "Holiday 20% Off",           "percent",       20, 100),
    (6, "NEWUSER10",   "New Customer 10% Off",      "percent",       10, 0),
    (7, "BFRIDAY30",   "Black Friday 30% Off",      "percent",       30, 0),
    (8, "CYBERMON",    "Cyber Monday $75 Off",      "fixed",         75, 300),
    (9, "BACK2SCHOOL", "Back to School 12% Off",    "percent",       12, 0),
    (10,"LOYALTY25",   "Loyalty Member $25 Off",    "fixed",         25, 150),
]

## Azure SQL connection

Uses an AAD access token from `notebookutils.credentials` so the JDBC connection runs as the user who triggered the notebook (the deployer, who is SQL admin).

In [ ]:
# notebookutils is pre-bound as a global in Fabric notebooks (no import required).
# Get an AAD token for Azure SQL on behalf of the user who triggered this run.
sql_token = notebookutils.credentials.getToken("https://database.windows.net/")

jdbc_url = (
    f"jdbc:sqlserver://{sql_server_fqdn}:1433;"
    f"database={sql_database_name};"
    "encrypt=true;trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;loginTimeout=30;"
)

jdbc_props = {
    "accessToken": sql_token,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

def write_table(df, table_fqn, mode="append"):
    """Write a Spark DataFrame to Azure SQL via JDBC."""
    (df.write
       .format("jdbc")
       .option("url", jdbc_url)
       .option("dbtable", table_fqn)
       .options(**jdbc_props)
       .mode(mode)
       .save())
    print(f"  wrote {df.count():,} rows -> {table_fqn}")


In [ ]:
# Idempotency: DELETE child tables first (FK-respecting order).
# Use Sparks JVM gateway to call the MSSQL JDBC driver directly (already on
# the classpath because the JDBC writer above uses it). Avoids needing
# jaydebeapi/pyodbc/pymssql -- none ship in the default Fabric Spark runtime.
def exec_sql(stmts):
    gw = spark.sparkContext._gateway
    props = gw.jvm.java.util.Properties()
    props.setProperty("accessToken", sql_token)
    conn = gw.jvm.java.sql.DriverManager.getConnection(jdbc_url, props)
    try:
        stmt = conn.createStatement()
        try:
            for s in stmts:
                stmt.execute(s)
        finally:
            stmt.close()
    finally:
        conn.close()

exec_sql([
    "DELETE FROM retail.reviews",
    "DELETE FROM retail.returns",
    "DELETE FROM retail.shipments",
    "DELETE FROM retail.payments",
    "DELETE FROM retail.order_items",
    "DELETE FROM retail.orders",
    "DELETE FROM retail.inventory",
    "DELETE FROM retail.promotions",
    "DELETE FROM retail.customers",
    "DELETE FROM retail.products",
    "DELETE FROM retail.stores",
    "DELETE FROM retail.warehouses",
    "DELETE FROM retail.suppliers",
    "DELETE FROM retail.brands",
    "DELETE FROM retail.categories",
])
print("Truncated all retail.* tables")


## Load reference data

In [ ]:
cats_df = spark.createDataFrame(
    [Row(category_id=c[0], parent_category_id=c[1], category_name=c[2], category_path=c[3], sort_order=c[4]) for c in CATEGORIES]
)
write_table(cats_df, "retail.categories")

brands_df = spark.createDataFrame(
    [Row(brand_id=b[0], brand_name=b[1], country_of_origin=b[2], is_premium=b[3]) for b in BRANDS]
)
write_table(brands_df, "retail.brands")

supp_df = spark.createDataFrame(
    [Row(supplier_id=s[0], supplier_name=s[1], contact_email=s[2], country=s[3], lead_time_days=s[4]) for s in SUPPLIERS]
)
write_table(supp_df, "retail.suppliers")

wh_df = spark.createDataFrame(
    [Row(warehouse_id=w[0], warehouse_name=w[1], city=w[2], state=w[3], country=w[4], capacity_units=w[5]) for w in WAREHOUSES]
)
write_table(wh_df, "retail.warehouses")

stores_df = spark.createDataFrame([
    Row(store_id=s[0], store_name=s[1], store_type=s[2], address_line1=s[3], city=s[4], state=s[5],
        postal_code=s[6], country=s[7], opened_at=datetime.date.fromisoformat(s[8]), square_feet=s[9], manager_name=s[10])
    for s in STORES
])
write_table(stores_df, "retail.stores")

# promotions: explicit schema because usage_limit is None for every row,
# so Spark cant infer the column type from the data alone.
promo_schema = StructType([
    StructField("promo_code",       StringType(),    False),
    StructField("promo_name",       StringType(),    False),
    StructField("discount_type",    StringType(),    False),
    StructField("discount_value",   DoubleType(),    False),
    StructField("min_order_amount", DoubleType(),    False),
    StructField("starts_at",        TimestampType(), False),
    StructField("ends_at",          TimestampType(), False),
    StructField("usage_limit",      IntegerType(),   True),
    StructField("times_used",       IntegerType(),   False),
])
promo_rows = [
    (p[1], p[2], p[3], float(p[4]), float(p[5]),
     datetime.datetime.combine(seed_start, datetime.time(0)),
     datetime.datetime.combine(seed_end, datetime.time(23,59,59)),
     None, 0)
    for p in PROMOTIONS
]
promo_df = spark.createDataFrame(promo_rows, schema=promo_schema)
write_table(promo_df, "retail.promotions")


## Products

In [ ]:
PRODUCT_TEMPLATES = [
    # (category_id, name_prefix, brand_ids, price_range, cost_frac, colors, warranty_months)
    (2, "Smartphone Pro",   [1,2,5,9], (799,1499), 0.55, ["Black","Silver","Blue"],         24),
    (3, "Laptop Ultra",     [1,4,6,8], (999,2799), 0.60, ["Space Gray","Silver"],           24),
    (4, "Tablet Air",       [1,2,5],   (449,1199), 0.55, ["Black","White","Rose"],          12),
    (5, "Headphones Studio",[3,7,9],   (149,499),  0.45, ["Black","White","Sand"],          12),
    (6, "Smart Watch",      [1,5,9],   (199,799),  0.50, ["Black","Silver","Gold"],         12),
    (7, "Mirrorless Camera",[3,4,9],   (699,2999), 0.65, ["Black"],                          24),
    (8, "Smart Speaker",    [1,2,10],  (49,349),   0.40, ["Charcoal","Sand","Forest"],      12),
    (10,"Gaming Console",   [4,6,10],  (399,599),  0.70, ["Black","White"],                  12),
    (11,"Game Controller",  [4,6,10],  (39,89),    0.35, ["Black","White","Red","Blue"],     6),
    (12,"USB-C Cable 6ft",  [3,8],     (9,29),     0.20, ["Black","White"],                  6),
    (12,"Wireless Charger", [1,3,8],   (29,79),    0.30, ["Black","White"],                  12),
    (12,"Laptop Stand",     [6,7],     (39,129),   0.30, ["Silver","Black"],                 12),
]

import math
tmpls = PRODUCT_TEMPLATES * math.ceil(n_products / len(PRODUCT_TEMPLATES))
products = []
for i, tmpl in enumerate(tmpls[:n_products], start=1):
    cat_id, name_prefix, brand_ids, price_range, cost_frac, colors, warranty = tmpl
    brand_id = random.choice(brand_ids)
    color    = random.choice(colors)
    price    = round(random.uniform(*price_range), 2)
    cost     = round(price * cost_frac, 2)
    year     = random.randint(2020, seed_end.year)
    brand_nm = next(b[1] for b in BRANDS if b[0] == brand_id)
    name     = f"{brand_nm} {name_prefix} {color} ({year})"
    products.append(Row(
        sku=f"CT-{i:06d}", product_name=name, description=None, category_id=cat_id,
        brand_id=brand_id, supplier_id=random.randint(1, len(SUPPLIERS)),
        list_price=price, cost=cost, weight_kg=round(random.uniform(0.05,4.5),3),
        dimensions_cm=None, color=color, model_year=year,
        upc=str(random.randint(10**11, 10**12 - 1)), warranty_months=warranty,
        is_active=True,
        launched_at=fake.date_between(start_date=datetime.date(2020,1,1), end_date=seed_end),
        discontinued_at=None,
    ))

products_df = spark.createDataFrame(products)
write_table(products_df, "retail.products")

## Customers

In [ ]:
loyalty_tiers   = ["Bronze","Silver","Gold","Platinum"]
loyalty_weights = [0.60, 0.25, 0.12, 0.03]

def gen_customer(i):
    first = fake.first_name()
    last  = fake.last_name()
    eu    = uuid.uuid4().hex[:8]
    created = fake.date_time_between(start_date="-5y", end_date=seed_start)
    return Row(
        email=f"{first.lower()}.{last.lower()}.{eu}@{fake.free_email_domain()}",
        first_name=first, last_name=last,
        phone=fake.numerify("###-###-####"),
        date_of_birth=fake.date_of_birth(minimum_age=18, maximum_age=80),
        segment_id=random.choices([1,2,3,4], weights=[60,25,10,5])[0],
        address_line1=fake.street_address(), address_line2=None,
        city=fake.city(), state=fake.state_abbr(), postal_code=fake.zipcode(), country="USA",
        loyalty_tier=random.choices(loyalty_tiers, weights=loyalty_weights)[0],
        loyalty_points=random.randint(0,50000),
        marketing_opt_in=random.random() < 0.55,
        created_at=created,
        last_login_at=fake.date_time_between(start_date=created, end_date=seed_end) if random.random() < 0.8 else None,
        is_active=True,
    )

customers = [gen_customer(i) for i in range(n_customers)]
cust_df = spark.createDataFrame(customers)
write_table(cust_df, "retail.customers")

## Inventory

In [ ]:
inv_rows = []
for pid in range(1, n_products + 1):
    for wh in WAREHOUSES:
        inv_rows.append(Row(
            product_id=pid, location_type="warehouse", location_id=wh[0],
            quantity_on_hand=random.randint(0,500), quantity_reserved=0,
            reorder_point=random.randint(5,50), reorder_quantity=random.randint(25,200),
            last_restocked_at=fake.date_time_between(start_date=seed_start, end_date=seed_end),
        ))
inv_df = spark.createDataFrame(inv_rows)
write_table(inv_df, "retail.inventory")

## Orders + items + payments + shipments

We need real surrogate ids back from SQL for FK chaining (orders.order_id, order_items.order_item_id). Strategy:

1. INSERT orders, then SELECT them back keyed by `order_number` to get assigned `order_id`s
2. Build items / payments / shipments using those ids
3. INSERT items / payments / shipments
4. SELECT items back to get assigned `order_item_id`s for returns / reviews

In [ ]:
channels         = ["online","store","mobile"]
channel_weights  = [0.60, 0.20, 0.20]
payment_methods  = ["credit_card","debit_card","paypal","apple_pay","google_pay","store_credit","gift_card"]
payment_weights  = [0.45,0.20,0.15,0.08,0.05,0.04,0.03]
card_brands      = ["Visa","Mastercard","Amex","Discover",None,None,None]
carriers         = ["UPS","FedEx","USPS","DHL"]
order_statuses   = ["delivered","delivered","delivered","shipped","paid","cancelled"]
status_weights   = [55,15,5,10,10,5]

product_prices = {p['sku']: p['list_price'] for p in products}
skus = list(product_prices.keys())

orders_in = []  # holds (order_number, planned items) for later
for i in range(n_orders):
    on  = f"ORD-{i+1:010d}"
    cust_id = random.randint(1, n_customers)
    channel = random.choices(channels, weights=channel_weights)[0]
    store_id = random.randint(1, len(STORES)) if channel == "store" else None
    promo_id = random.choice([None,None,None] + [p[0] for p in PROMOTIONS])
    odate = fake.date_time_between(start_date=seed_start, end_date=seed_end)
    n_items = random.choices([1,2,3,4,5], weights=[40,30,15,10,5])[0]

    items = []
    subtotal = 0.0; discount = 0.0
    for _ in range(n_items):
        sku = random.choice(skus)
        list_price = product_prices[sku]
        qty = random.randint(1,3)
        unit_price = round(list_price * random.uniform(0.85, 1.0), 2)
        line_gross = round(unit_price * qty, 2)
        line_disc  = round(line_gross * (0.1 if promo_id else 0), 2)
        line_total = round(line_gross - line_disc, 2)
        subtotal += line_gross; discount += line_disc
        items.append((sku, qty, unit_price, line_disc, line_total, random.randint(1, len(WAREHOUSES))))

    tax = round((subtotal - discount) * 0.08, 2)
    shipping = 0.0 if (subtotal - discount) > 99 else round(random.uniform(5.99, 14.99), 2)
    total = round(subtotal - discount + tax + shipping, 2)
    status = random.choices(order_statuses, weights=status_weights)[0]

    orders_in.append({
        "order_number": on, "customer_id": cust_id, "order_date": odate,
        "order_status": status, "channel": channel, "store_id": store_id,
        "subtotal": round(subtotal,2), "tax_amount": tax, "shipping_amount": shipping,
        "discount_amount": round(discount,2), "total_amount": total, "currency": "USD",
        "promotion_id": promo_id,
        "ship_address_line1": fake.street_address(), "ship_city": fake.city(),
        "ship_state": fake.state_abbr(), "ship_postal_code": fake.zipcode(), "ship_country": "USA",
        "_items": items,
    })

print(f"Prepared {len(orders_in):,} orders in driver memory")

In [ ]:
# Insert orders (without items yet)
orders_only = [{k:v for k,v in o.items() if k != "_items"} for o in orders_in]
orders_df = spark.createDataFrame([Row(**o) for o in orders_only])
write_table(orders_df, "retail.orders")

# Read back to get the IDENTITY-assigned order_id
order_ids_df = (spark.read.format("jdbc")
    .option("url", jdbc_url)
    .option("dbtable", "(SELECT order_id, order_number FROM retail.orders) t")
    .options(**jdbc_props)
    .load())
order_id_map = {r['order_number']: r['order_id'] for r in order_ids_df.collect()}
print(f"Resolved {len(order_id_map):,} order_id values")

In [ ]:
# Now build items / payments / shipments using the resolved order_ids
sku_to_pid = {p['sku']: idx+1 for idx, p in enumerate(products)}

item_rows = []; pay_rows = []; ship_rows = []
for o in orders_in:
    oid = order_id_map[o['order_number']]
    for (sku, qty, unit_price, line_disc, line_total, wh_id) in o['_items']:
        item_rows.append(Row(order_id=oid, product_id=sku_to_pid[sku], quantity=qty,
                             unit_price=unit_price, line_discount=line_disc,
                             line_total=line_total, fulfillment_warehouse_id=wh_id))
    # Payment
    pm = random.choices(payment_methods, weights=payment_weights)[0]
    cb = random.choice(card_brands) if "card" in pm else None
    cl4 = fake.numerify("####") if cb else None
    pay_status = "captured" if o['order_status'] != "cancelled" else "voided"
    pay_rows.append(Row(order_id=oid, payment_method=pm, card_brand=cb, card_last_four=cl4,
                        amount=o['total_amount'], status=pay_status,
                        transaction_ref=uuid.uuid4().hex.upper()[:20], processed_at=o['order_date']))
    # Shipment for non-store, non-cancelled
    if o['channel'] != "store" and o['order_status'] not in ("cancelled","paid"):
        carrier = random.choice(carriers)
        tracking = f"{carrier[:3].upper()}{uuid.uuid4().hex[:12].upper()}"
        wh_id   = random.randint(1, len(WAREHOUSES))
        shipped = fake.date_time_between(start_date=o['order_date'], end_date=seed_end) if o['order_status'] in ("shipped","delivered") else None
        est_del = (shipped.date() + datetime.timedelta(days=random.randint(2,7))) if shipped else None
        delivered = fake.date_time_between(start_date=shipped, end_date=seed_end) if (o['order_status']=="delivered" and shipped) else None
        ship_status = {"delivered":"delivered","shipped":"in_transit","paid":"label_created"}.get(o['order_status'], "label_created")
        ship_rows.append(Row(order_id=oid, warehouse_id=wh_id, carrier=carrier, tracking_number=tracking,
                             shipped_at=shipped, estimated_delivery=est_del, delivered_at=delivered, status=ship_status))

print(f"Built {len(item_rows):,} items, {len(pay_rows):,} payments, {len(ship_rows):,} shipments")

In [ ]:
items_df    = spark.createDataFrame(item_rows)
payments_df = spark.createDataFrame(pay_rows)
shipments_df= spark.createDataFrame(ship_rows)
write_table(items_df, "retail.order_items")
write_table(payments_df, "retail.payments")
write_table(shipments_df, "retail.shipments")

## ADLS Gen2: dated daily files (CSV + Parquet) in `raw/`

Two source feeds:

- `raw/supplier_inventory/YYYY-MM-DD.parquet` — daily supplier on-hand snapshot
- `raw/marketing_campaigns/YYYY-MM-DD.csv` — daily marketing campaign performance

These mimic vendor / partner data drops landing in your data lake.

In [ ]:
raw_base = f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net"

# Supplier inventory: 1 row per (supplier, sku-group) per day, ~few thousand rows/day
supplier_skus = [s for s in skus[:200]]  # subset that suppliers actually quote on

for d_offset in range((seed_end - seed_start).days + 1):
    day = seed_start + datetime.timedelta(days=d_offset)
    rows = []
    for sku in supplier_skus:
        for supp_id in range(1, len(SUPPLIERS)+1):
            if random.random() < 0.4:  # not every supplier quotes every sku
                rows.append(Row(
                    snapshot_date=day, sku=sku, supplier_id=supp_id,
                    qty_available=random.randint(0, 2000),
                    unit_cost=round(random.uniform(5, 800), 2),
                    lead_time_days=random.randint(3, 30),
                ))
    if rows:
        df = spark.createDataFrame(rows)
        path = f"{raw_base}/supplier_inventory/{day.isoformat()}.parquet"
        df.coalesce(1).write.mode("overwrite").parquet(path)

print(f"Wrote supplier_inventory parquet files for {(seed_end-seed_start).days+1} days")

In [ ]:
# Marketing campaigns: 1 CSV per day with per-channel ad spend + impressions
campaign_channels = ["google_ads","meta_ads","tiktok","email","influencer","display"]

for d_offset in range((seed_end - seed_start).days + 1):
    day = seed_start + datetime.timedelta(days=d_offset)
    rows = []
    for ch in campaign_channels:
        impressions = random.randint(5_000, 250_000)
        clicks      = int(impressions * random.uniform(0.005, 0.04))
        spend       = round(impressions / 1000 * random.uniform(2.5, 18.0), 2)
        conversions = int(clicks * random.uniform(0.01, 0.08))
        rows.append(Row(
            campaign_date=day, channel=ch,
            impressions=impressions, clicks=clicks,
            spend_usd=spend, conversions=conversions,
        ))
    df = spark.createDataFrame(rows)
    path = f"{raw_base}/marketing_campaigns/{day.isoformat()}.csv"
    df.coalesce(1).write.mode("overwrite").option("header", True).csv(path)

print(f"Wrote marketing_campaigns csv files for {(seed_end-seed_start).days+1} days")

## Done

Azure SQL is populated with a full fiscal quarter; ADLS `raw/` has 90+ days of supplier Parquet + marketing CSV files.

`deploy.ps1` will now set up the SQL Mirror and the ADLS Shortcut against this populated data.

In [ ]:
notebookutils.notebook.exit("OK")
